# ① 전처리 — 문서 파싱 · 청크 분할
담당: 전처리 · 코드: `rag/preprocessing/parsing.py`, `rag/preprocessing/chunking.py`

`config.yaml` 의 `paths.input`(ZIP 또는 폴더)에 있는 PDF · HWP · HWPX 를 읽어 본문 · 표 · (선택) 그림 설명을 뽑고, 청크로 나눕니다.

| 저장 파일 | 읽는 노트북 |
|---|---|
| `artifacts/chunks.jsonl` (청크) | 02 검색, 03 검증 |
| `artifacts/docs.jsonl` (문서 목록 — 발주기관 라우팅용) | 02 검색 |

## 설정
`OCR = True` 로 바꾸면 문서 속 그림을 OpenAI 비전 모델(`gpt-5-mini`, `.env` 의 `OPENAI_API_KEY`)로 글로 바꿔 청크에 넣습니다 (유료).
로고 · 장식처럼 여러 쪽에 반복되는 그림(`config.yaml` 의 `max_repeat_image_pages` 쪽 이상)은 청크에 넣지 않습니다.
`False` 면 그림은 API 로 보내지 않습니다.

In [6]:
OCR = True

In [7]:
from rag.config import ARTIFACTS, ROOT, load_config
from rag.preprocessing.chunking import chunk_documents, chunk_summary, save_chunks
from rag.preprocessing.parsing import collect_inputs, parse_documents, parsing_summary

cfg = load_config()

## 1. 입력 파일 모으기
ZIP(안의 ZIP 포함)을 `artifacts/raw/` 에 풀고 PDF · HWP · HWPX 목록을 만듭니다.

In [8]:
files = collect_inputs(ROOT / cfg["paths"]["input"], ARTIFACTS / "raw")
len(files)

98

## 2. 파싱
문서마다 쪽 · 표 · 그림 수와 표지 감지 결과를 봅니다.

In [9]:
docs = parse_documents(files, cfg["parsing"], ocr=OCR)
parsing_summary(docs)

(1/98) (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp: 블록 299개
(2/98) (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp: 블록 155개
(3/98) (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp: 블록 205개
(4/98) (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp: 블록 145개
(5/98) 2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp: 블록 133개
(6/98) BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).hwp: 블록 207개
(7/98) KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp: 블록 281개
(8/98) 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp: 블록 176개
(9/98) 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp: 블록 192개
(10/98) 경기도사회서비스원_2024년 통합사회정보시스템 운영지원.hwp: 블록 163개
(11/98) 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp: 블록 163개
(12/98) 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp: 블록 81개
(13/98) 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf: 블록 1031개
(14/98) 고양도시관리공사_관산근린공원 다목적구장 홈페이지 및 회원 통합운영.hwp: 블록 191개
(15/98) 광주과학기술원_실시간통합연구비관리시스템(RCMS)  연계 모듈 변경 사업.hwp: 블록 157개
(16/98) 광주과학기술원_학사시스템 기능개선 사업.hwp: 블록 135개
(17/98) 국가과학기술지식정보서비스_통합정보시스템 고도화 용역.hwp: 블록 192개
(18/98) 국가철도공단_철도인프라 디지털트윈 정보

,문서,형식,text,table,image_caption,글자 수
0,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp,hwp,77,214,8.0,123438
1,(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원...,hwp,39,101,15.0,63490
2,(사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp,hwp,58,142,5.0,87305
3,(재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp,hwp,56,83,6.0,55480
4,2025 구미 아시아육상경기선수권대회 조직위원회_2025 구미아시아육상경.hwp,hwp,50,83,0.0,54417
...,...,...,...,...,...,...
93,한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp,hwp,84,152,5.0,126292
94,한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp,hwp,80,224,4.0,156272
95,한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp,hwp,71,107,8.0,88022
96,한국해양조사협회_2024년 항해용 간행물 품질관리 업무보조 시스템 구축.hwp,hwp,60,75,7.0,70770


## 3. 청크 분할 · 저장

In [10]:
chunks = chunk_documents(docs, cfg["chunking"])
save_chunks(docs, chunks, ARTIFACTS)
chunk_summary(chunks)

,청크 수,text,table,image_caption,평균 길이,중앙값,문장 중간 절단(%),섹션 부착(%)
0,18630,4994,12620,1016,469.9,442,45.1,93.9
